In [0]:
SUPABASE_HOST     = "aws-1-us-east-1.pooler.supabase.com"
SUPABASE_PORT     = "6543"
SUPABASE_DB       = "postgres"
SUPABASE_USER     = "postgres.yhiuptcspzqeliwaopxv"
SUPABASE_PASSWORD = "trabalhoarquiteturasuperdificil"  # a senha do projeto

JDBC_URL = (
    f"jdbc:postgresql://{SUPABASE_HOST}:{SUPABASE_PORT}/{SUPABASE_DB}"
    "?sslmode=require"
)

JDBC_PROPS = {
    "user":     SUPABASE_USER,
    "password": SUPABASE_PASSWORD,
    "driver":   "org.postgresql.Driver"
}

SCHEMA_LANDING = "landing"
CATALOG        = "workspace"   # Unity Catalog catalog name (Free Edition)

print(f"JDBC URL: {JDBC_URL}")
print("Parâmetros configurados.")


JDBC URL: jdbc:postgresql://aws-1-us-east-1.pooler.supabase.com:6543/postgres?sslmode=require
Parâmetros configurados.


In [0]:
# ===========================================================================
# CÉLULA 2 - Criar schema LANDING no Databricks
# ===========================================================================
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_LANDING}")
print(f"Schema '{SCHEMA_LANDING}' criado/verificado com sucesso.")

Schema 'landing' criado/verificado com sucesso.


In [0]:
# ===========================================================================
# CÉLULA 3 - Lista de tabelas para extrair
# ===========================================================================
TABELAS = [
    "clientes",
    "categorias",
    "produtos",
    "pedidos",
    "itens_pedido"
]
print(f"Tabelas a extrair: {TABELAS}")

Tabelas a extrair: ['clientes', 'categorias', 'produtos', 'pedidos', 'itens_pedido']


In [0]:
# ===========================================================================
# CÉLULA 4 - Loop de extração: Supabase → CSV no Databricks (Volume)
# ===========================================================================
# Usaremos Databricks Volumes (disponível no Free Edition) como "landing zone"
# Caminho: /Volumes/<catalog>/<schema>/dados/

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA_LANDING}/dados"

# Cria o volume se não existir (execute UMA vez)
spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA_LANDING}.dados
""")

resultados = {}

for tabela in TABELAS:
    print(f"\n>>> Extraindo tabela: {tabela}")
    try:
        df = spark.read \
            .format("jdbc") \
            .option("url", JDBC_URL) \
            .option("dbtable", f"public.{tabela}") \
            .option("user", SUPABASE_USER) \
            .option("password", SUPABASE_PASSWORD) \
            .option("driver", "org.postgresql.Driver") \
            .load()

        qtd = df.count()
        print(f"    Registros lidos: {qtd}")

        # Grava em CSV no Volume (landing zone)
        caminho_saida = f"{VOLUME_PATH}/{tabela}"
        df.coalesce(1).write \
            .mode("overwrite") \
            .option("header", "true") \
            .csv(caminho_saida)

        print(f"    Salvo em: {caminho_saida}")
        resultados[tabela] = {"status": "OK", "registros": qtd}

    except Exception as e:
        print(f"    ERRO: {e}")
        resultados[tabela] = {"status": "ERRO", "mensagem": str(e)}


>>> Extraindo tabela: clientes
    Registros lidos: 10
    Salvo em: /Volumes/workspace/landing/dados/clientes

>>> Extraindo tabela: categorias
    Registros lidos: 5
    Salvo em: /Volumes/workspace/landing/dados/categorias

>>> Extraindo tabela: produtos
    Registros lidos: 10
    Salvo em: /Volumes/workspace/landing/dados/produtos

>>> Extraindo tabela: pedidos
    Registros lidos: 12
    Salvo em: /Volumes/workspace/landing/dados/pedidos

>>> Extraindo tabela: itens_pedido
    Registros lidos: 20
    Salvo em: /Volumes/workspace/landing/dados/itens_pedido


In [0]:
# ===========================================================================
# CÉLULA 5 - Relatório de extração
# ===========================================================================
print("\n" + "="*50)
print("RELATÓRIO DE EXTRAÇÃO - LANDING")
print("="*50)
for tabela, info in resultados.items():
    status = info.get("status")
    regs   = info.get("registros", "-")
    print(f"  {tabela:<20} | Status: {status:<5} | Registros: {regs}")



RELATÓRIO DE EXTRAÇÃO - LANDING
  clientes             | Status: OK    | Registros: 10
  categorias           | Status: OK    | Registros: 5
  produtos             | Status: OK    | Registros: 10
  pedidos              | Status: OK    | Registros: 12
  itens_pedido         | Status: OK    | Registros: 20


In [0]:
# ===========================================================================
# CÉLULA 6 - Listar arquivos gerados
# ===========================================================================
dbutils.fs.ls(VOLUME_PATH)

JDBC_PROPS = {
    "user":     SUPABASE_USER,
    "password": SUPABASE_PASSWORD,
    "driver":   "org.postgresql.Driver"
}

SCHEMA_LANDING = "landing"
CATALOG        = "workspace"   # Unity Catalog catalog name (Free Edition)

print(f"JDBC URL: {JDBC_URL}")
print("Parâmetros configurados.")

JDBC URL: jdbc:postgresql://aws-1-us-east-1.pooler.supabase.com:6543/postgres?sslmode=require
Parâmetros configurados.
